In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.ticker import MaxNLocator, MultipleLocator

from tempfile import NamedTemporaryFile

In [ ]:


input_path = Path("derived/intermediate") / "prepared_dataset.xlsx"  # Path to the input workbook.
output_dir = Path("distribution")  # Folder for generated results.
plot_dpi = 600  # Distribution figure resolution.
support_columns = ["Support1", "Support2", "Support3"]  
required_columns = ["T", "P", "H2/CO2", "GHSV", "Method", *support_columns]  

data = pd.read_excel(input_path, sheet_name="data", engine="openpyxl")  # Input records for this analysis.
missing_columns = set(required_columns) - set(data.columns)  
if missing_columns:
    raise ValueError(f"Missing columns: {sorted(missing_columns)}")

conditions = data[["T", "P", "H2/CO2", "GHSV"]].apply(pd.to_numeric)  # Reaction conditions used in the analysis.
if conditions.empty or not np.isfinite(conditions.to_numpy()).all():
    raise ValueError("Reaction conditions must contain finite numeric values.")

output_dir.mkdir(exist_ok=True)
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": 11,
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "mathtext.default": "bf",
    "axes.linewidth": 1.1,
    "xtick.major.width": 1.1,
    "ytick.major.width": 1.1,
    "savefig.facecolor": "white",
})

In [ ]:
histogram_settings = {  
    "T": (400, 675, 25, 50, "Temperature (K)"),
    "P": (0, 10, 1, 1, "Pressure (MPa)"),
    "GHSV": (0, 900, 100, 200, r"GHSV ($10^2$ cm$^3$ h$^{-1}$ g$_{\mathrm{cat}}^{-1}$)"),
    "H2/CO2": (0.5, 10.5, 1, 1, r"H$_2$/CO$_2$ feed ratio"),
}
histograms = {}  

for column, (lower, upper, width, _, _) in histogram_settings.items():  # Current column name; Lower bound for the current interval; Upper bound on predicted values; Width.
    values = conditions[column] / 100 if column == "GHSV" else conditions[column]  # Values used in the current calculation.
    lower = min(lower, lower + np.floor((values.min() - lower) / width) * width)
    upper = max(upper, lower + (np.floor((values.max() - lower) / width) + 1) * width)
    edges = np.arange(lower, upper + width / 2, width)  
    counts, _ = np.histogram(values, bins=edges)  
    if counts.sum() != len(values):
        raise ValueError(f"The histogram for {column} does not cover all observations.")
    histograms[column] = (edges, counts)

method_abbreviations = {  
    "co-precripitation": "CP",
    "co-precipitation": "CP",
    "dry impregnation": "DI",
    "wet impregnation": "WI",
    "deposition-precipitation": "DP",
    "hydrothermal synthesis": "HT",
    "sol gel": "SG",
    "solvothermal": "ST",
    "microwave assisted": "MWA",
    "other": "Other",
}
method_labels = data["Method"].astype("string").str.strip().str.lower()  
method_counts = method_labels.replace(method_abbreviations).replace("", pd.NA).dropna().value_counts()  

support_values = pd.concat([data[column] for column in support_columns], ignore_index=True)  # Support values. Current column name.
support_values = support_values.dropna().astype("string").str.strip()
support_counts = support_values[~support_values.isin(["", "0", "0.0", "0.00"])].value_counts()  
support_plot_counts = support_counts.head(5).copy()  
if len(support_counts) > 5:
    support_plot_counts.loc["Other"] = support_counts.iloc[5:].sum()

if method_counts.empty or support_counts.empty:
    raise ValueError("The workbook needs synthesis methods and support materials to plot.")

display(conditions.describe().T)
for column, (edges, counts) in histograms.items():
    print(histogram_settings[column][-1] if column != "GHSV" else "GHSV (100 cm³/h/gcat)")
    display(pd.DataFrame({"Lower edge": edges[:-1], "Upper edge": edges[1:], "Count": counts}))

display(support_plot_counts.rename("Support occurrences").to_frame())
display(pd.DataFrame({
    "Method count": method_counts,
    "Share (%)": (100 * method_counts / method_counts.sum()).round(1),
}))

In [ ]:
table_s2_features = {  
    "Catalyst composition": [
        ("wt% Cu", "Cu loading (wt%)"),
        ("wt% Support1", "Support 1 loading (wt%)"),
        ("χ Support1", "χ Support 1 (eV)"),
        ("wt% Support2", "Support 2 loading (wt%)"),
        ("χ Support2", "χ Support 2 (eV)"),
        ("wt% Support3", "Support 3 loading (wt%)"),
        ("χ Support3", "χ Support 3 (eV)"),
        ("wt% Promoter", "Promoter loading (wt%)"),
        ("χ Promoter", "χ Promoter (eV)"),
    ],
    "Pretreatment conditions": [
        ("Calc T", "Calcination temperature (K)"),
        ("Calc D", "Calcination duration (h)"),
        ("Synthesis Method", "Synthesis method"),
    ],
    "Reaction conditions": [
        ("H2/CO2", "H₂/CO₂"),
        ("T", "Temperature (K)"),
        ("P", "Pressure (MPa)"),
        ("GHSV", "GHSV (cm³ h⁻¹ gcat⁻¹)"),
    ],
    "Characterization results": [
        ("dCu", "dCu (nm)"),
        ("SBET", "SBET (m² gcat⁻¹)"),
        ("SACu", "SACu (m² gcat⁻¹)"),
    ],
}
calcination_flags = {"Calc T": "Calc T imputed", "Calc D": "Calc D imputed"}  
table_s2_rows = []  
observed_calcination = {}  

for category, features_with_units in table_s2_features.items():  
    for index, (column, label) in enumerate(features_with_units):  # Current position or index label
        values = data[column]
        if column in calcination_flags:
            flags = data[calcination_flags[column]]  
            if not flags.isin([True, False]).all():
                raise ValueError(f"Invalid imputation flags for {column}.")
            values = values.mask(flags.astype(bool))
            observed_calcination[column] = values.count()

        if column == "Synthesis Method":
            values = values.astype("string").str.strip().str.lower().replace("", pd.NA)
            average, deviation, median, minimum, maximum = [np.nan] * 5 
        else:
            values = pd.to_numeric(values)
            average, deviation, median, minimum, maximum = values.agg(["mean", "std", "median", "min", "max"])
        mode = values.mode()  
        table_s2_rows.append([
            category if index == 0 else "", label, average, deviation, median,
            mode.iat[0] if not mode.empty else None, minimum, maximum,
        ])

table_s2 = pd.DataFrame(table_s2_rows, columns=[  
    "Category", "Feature", "Average", "Standard Deviation", "Median", "Mode", "Minimum", "Maximum",
])
table_s2_title = "Table S2. Descriptive statistics of features including the average, standard deviation, median value, mode value, minimum, and maximum."  
table_s2_note = (  
    f"Statistics are based on {len(data)} screened records. Calcination statistics use observed values before "
    f"imputation (temperature: {observed_calcination['Calc T']}, duration: {observed_calcination['Calc D']}). "
    "For synthesis method, the mode is based on the grouped categories used in modeling."
)
print(table_s2_title)
display(table_s2.style.hide(axis="index").format(precision=2, na_rep="—"))  # Display Table S2, descriptive statistics for the modeling features.
print(table_s2_note)


In [ ]:
bar_style = {"color": "#FFB347", "edgecolor": "black", "linewidth": 0.6}  
method_colors = {  
    "CP": "#d62728", "DP": "#ff7f0e", "WI": "#2ca02c", "DI": "#4E79A7",
    "SG": "#9467bd", "HT": "#8c564b", "ST": "#e377c2", "MWA": "#7f7f7f",
    "Other": "#EDC949",
}


def draw_histogram(fig, slot, column):  # Draw a distribution histogram for one reaction variable.
    ax = fig.add_subplot(slot)  # Plot axes.
    edges, counts = histograms[column]  
    _, _, _, tick_width, label = histogram_settings[column] 
    ax.bar(edges[:-1], counts, width=np.diff(edges), align="edge", **bar_style)
    ax.set(xlabel=label, ylabel="Frequency", xlim=(edges[0], edges[-1]))
    ax.xaxis.set_major_locator(MultipleLocator(tick_width))
    if column == "T":
        ax.set_xlim(min(350, edges[0]), max(700, edges[-1]))
        ax.set_ylim(0, max(400, 50 * np.ceil(counts.max() / 50)))
        ax.yaxis.set_major_locator(MultipleLocator(50))
    else:
        ax.yaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))
    return ax


def draw_ratio(fig, slot):  # Draw the feed-ratio histogram with a broken frequency axis.
    edges, counts = histograms["H2/CO2"]  
    largest, second_largest = sorted(counts, reverse=True)[:2]  
    if largest < 100 or largest < 5 * max(second_largest, 1):
        return draw_histogram(fig, slot, "H2/CO2")

    slots = slot.subgridspec(2, 1, height_ratios=(1, 2.7), hspace=0.015)  
    top = fig.add_subplot(slots[0])  # Highest-ranked entries selected for display.
    bottom = fig.add_subplot(slots[1], sharex=top)  
    for ax in (top, bottom):  
        ax.bar(edges[:-1], counts, width=np.diff(edges), align="edge", **bar_style)
        ax.xaxis.set_major_locator(MultipleLocator(1))

    top.set_ylim(20 * (largest // 20 - 1) - 2, 20 * (largest // 20 + 2) + 2)
    bottom.set_ylim(0, 20 * (second_largest // 20 + 1) + 2)
    bottom.set_xlim(edges[0] - 0.3, edges[-1] + 0.3)
    bottom.set_xlabel(histogram_settings["H2/CO2"][-1])
    bottom.set_ylabel("Frequency")
    top.yaxis.set_major_locator(MultipleLocator(20))
    bottom.yaxis.set_major_locator(MultipleLocator(20))
    top.spines["bottom"].set_visible(False)
    bottom.spines["top"].set_visible(False)
    top.tick_params(axis="x", bottom=False, labelbottom=False)
    break_style = {  
        "marker": [(-1, -0.45), (1, 0.45)], "markersize": 8,
        "linestyle": "none", "color": "black", "clip_on": False,
    }
    top.plot([0, 1], [0, 0], transform=top.transAxes, **break_style)
    bottom.plot([0, 1], [1, 1], transform=bottom.transAxes, **break_style)
    return top


def draw_supports(fig, slot):  # Draw support occurrences across all three support fields.
    ax = fig.add_subplot(slot)  
    positions = np.arange(len(support_plot_counts))  
    ax.bar(positions, support_plot_counts.to_numpy(), **bar_style)
    ax.set(xlabel="Support", ylabel="Frequency")
    ax.set_xticks(positions, support_plot_counts.index, rotation=20, ha="right")
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))
    return ax


def draw_methods(fig, slot):  # Draw the synthesis-method frequency pie chart.
    slots = slot.subgridspec(1, 2, width_ratios=(3.7, 1), wspace=0.01)  
    ax = fig.add_subplot(slots[0])  
    legend_ax = fig.add_subplot(slots[1])  
    legend_ax.axis("off")
    colors = [method_colors.get(method, plt.get_cmap("tab20")(i % 20))  
              for i, method in enumerate(method_counts.index)]  # Loop position; Method.
    wedges, _, labels = ax.pie(  # Wedges; Group or display labels.
        method_counts,
        colors=colors,
        startangle=90,
        autopct=lambda _: "",
        pctdistance=0.7,
        textprops={"fontsize": 9},
    )
    for label, count in zip(labels[:4], method_counts.iloc[:4]):  
        label.set_text(f"{count / method_counts.sum():.1%}")
    legend_ax.legend(
        wedges, method_counts.index, loc="center left", frameon=False,
        fontsize=9, handlelength=1.0, handletextpad=0.5, borderaxespad=0,
    )
    ax.set(xlim=(-1.04, 1.04), ylim=(-1.04, 1.04), xlabel="Synthesis Method")
    return ax

In [ ]:
panels = [  
    ("T", "T_hist.png"),
    ("P", "P_hist.png"),
    ("H2/CO2", "Ratio_hist.png"),
    ("GHSV", "GHSV_hist.png"),
    ("Support", "Support_plot.png"),
    ("Method", "Synthesis-Method_pie.png"),
]
custom_plots = {"H2/CO2": draw_ratio, "Support": draw_supports, "Method": draw_methods}  


def draw_panel(fig, slot, name):  # Select the plotting routine for one distribution panel.
    if name in custom_plots:
        return custom_plots[name](fig, slot)
    return draw_histogram(fig, slot, name)


for name, filename in panels:  # Current item name; Filename.
    figure = plt.figure(figsize=(6, 4.8), layout="constrained") 
    draw_panel(figure, figure.add_gridspec(1, 1)[0, 0], name)
    figure.savefig(output_dir / filename, dpi=plot_dpi, bbox_inches="tight")  # Save Fig. 2a–f as separate files in temperature, pressure, feed-ratio, GHSV, support, and synthesis-method order.
    plt.close(figure)

figure = plt.figure(figsize=(12, 8), layout="constrained")
figure.get_layout_engine().set(rect=(0, 0, 1, 0.96), hspace=0.09)
grid = figure.add_gridspec(2, 3)  
panel_axes = [ 
    draw_panel(figure, grid[i // 3, i % 3], name)
    for i, (name, _) in enumerate(panels)  
]

figure.draw_without_rendering()
figure.set_layout_engine("none")
positions = [ax.get_position() for ax in panel_axes]  
column_left = [min(positions[column].x0, positions[column + 3].x0) for column in range(3)]  # Column left. Current column name.
label_gap = positions[2].x0 - column_left[2]  
label_x = [column_left[0] - label_gap, column_left[1] - label_gap, column_left[2]] 
label_y = [  
    max(position.y1 for position in positions[row:row + 3]) + 6 / (72 * figure.get_figheight())  
    for row in (0, 3)  
]
for i, label in enumerate("abcdef"):  # Loop position.
    figure.text(
        label_x[i % 3], label_y[i // 3], f"({label})",
        ha="left", va="bottom", fontsize=19, fontweight="bold",
    )

figure.savefig(
    output_dir / "all_subplots_adjusted.tif",  # Save Fig. 2a–f as a single figure.
    dpi=plot_dpi,
    bbox_inches="tight",
    pil_kwargs={"compression": "tiff_lzw"},
)
display(figure)
plt.close(figure)
print(f"Figures saved to: {output_dir.resolve()}")

In [ ]:

encoded = data.copy()  
method_column = "Synthesis Method"  
encoded_path = Path("derived") / "dataset.xlsx"  # Path to the encoded modeling workbook.

method_codes = {  
    "hydrothermal synthesis": 0,
    "impregnation": 1,
    "microwave assisted": 2,
    "other": 3,
    "precipitation": 4,
    "sol gel": 5,
    "solvothermal": 6,
}
unknown_code = -1  
original_method = encoded[method_column].astype("string")  # Original method.
encoded[f"{method_column} unencoded"] = original_method
encoded[method_column] = original_method.str.strip().str.lower().map(method_codes).fillna(unknown_code).astype(int)

codebook = pd.DataFrame(  # Synthesis method encoding lookup.
    list(method_codes.items()) + [("Unknown or missing", unknown_code)],
    columns=["Synthesis Method", "Code"],
)

with NamedTemporaryFile(dir=encoded_path.parent, suffix=".xlsx", delete=False) as temporary:  
    temporary_path = Path(temporary.name)  

try:
    with pd.ExcelWriter(temporary_path, engine="openpyxl") as writer:  # Excel writer for the result workbook.
        encoded.to_excel(writer, sheet_name="data", index=False)
        codebook.to_excel(writer, sheet_name="encoding", index=False)
    temporary_path.replace(encoded_path)
finally:
    temporary_path.unlink(missing_ok=True)

print(f"Saved {len(encoded)} records to: {encoded_path.resolve()}")